# Gold DQ Tests — Salutar Saúde

Suíte de testes de qualidade de dados para todas as tabelas da camada Gold (`homologacao.salutar_saude.gold_*`).

## Categorias de Teste

| Categoria | Descrição |
|---|---|
| `VOLUME` | Contagem de linhas dentro do intervalo esperado |
| `UNIQUENESS` | Chaves primárias sem duplicatas |
| `COMPLETENESS` | Ausência de NULLs em colunas obrigatórias |
| `INTEGRITY` | Chaves estrangeiras existem na tabela referenciada |
| `DOMAIN` | Valores dentro do conjunto permitido |
| `BUSINESS_RULE` | Consistência de colunas derivadas e regras de negócio |

## Severity

| Status | Descrição |
|---|---|
| ✅ `PASS` | Teste passou |
| ⚠️ `WARN` | Violação encontrada mas não bloqueia o pipeline |
| ❌ `FAIL` | Violação crítica — bloqueia na célula de sumário |

## Ordem de Execução

Execute as células em ordem. A célula de **Configuração** deve ser a primeira pois inicializa o framework.

In [0]:
from dataclasses import dataclass, field
from typing import List
from datetime import datetime

# ── Configuração ─────────────────────────────────────────────────────────────────
CATALOG = "homologacao"
SCHEMA  = "salutar_saude"

def tbl(name: str) -> str:
    """Retorna o nome completamente qualificado de uma tabela gold."""
    return f"{CATALOG}.{SCHEMA}.gold_{name}"

_run_ts = datetime.now()
print(f"Gold DQ Tests — Salutar Saúde")
print(f"Iniciado  : {_run_ts:%Y-%m-%d %H:%M:%S}")
print(f"Schema    : {CATALOG}.{SCHEMA}")

# ── Framework ─────────────────────────────────────────────────────────────────
@dataclass
class DQResult:
    table    : str
    test     : str
    category : str   # VOLUME | COMPLETENESS | UNIQUENESS | INTEGRITY | DOMAIN | BUSINESS_RULE
    severity : str   # FAIL | WARN
    status   : str   # PASS | FAIL | WARN
    detail   : str
    value    : object = None

_RESULTS: List[DQResult] = []   # limpa ao re-executar esta célula
_ICONS = {"PASS": "✅", "WARN": "⚠️ ", "FAIL": "❌"}

def _register(table, test, category, severity, passed, detail, value=None):
    status = "PASS" if passed else severity
    _RESULTS.append(DQResult(table, test, category, severity, status, detail, value))
    icon = _ICONS[status]
    print(f"  {icon} [{status:<4}]  {table:<40} {test}")
    if not passed:
        print(f"           ↳ {detail}")

def assert_zero(table, test, category, severity, sql, detail):
    """PASS quando a query retorna 0 violações."""
    n = spark.sql(sql).collect()[0][0]
    _register(table, test, category, severity, n == 0,
              f"{detail}  [n={n:,}]", n)

def assert_eq(table, test, category, severity, sql, expected, detail):
    """PASS quando o resultado == esperado."""
    actual = spark.sql(sql).collect()[0][0]
    _register(table, test, category, severity, actual == expected,
              f"{detail}  [esperado={expected:,}, encontrado={actual:,}]", actual)

def assert_between(table, test, category, severity, sql, lo, hi, detail):
    """PASS quando lo ≤ resultado ≤ hi."""
    actual = spark.sql(sql).collect()[0][0]
    _register(table, test, category, severity, lo <= actual <= hi,
              f"{detail}  [esperado=[{lo:,}..{hi:,}], encontrado={actual:,}]", actual)

def section(title: str):
    print(f"\n{'─'*75}\n  {title}\n{'─'*75}")

print("\n[OK] Framework DQ inicializado. Execute os próximos cells em ordem.")

Gold DQ Tests — Salutar Saúde
Iniciado  : 2026-07-04 21:12:15
Schema    : homologacao.salutar_saude

[OK] Framework DQ inicializado. Execute os próximos cells em ordem.


In [0]:
section("dim_data")
T = tbl("dim_data")

# ─ Volume ─────────────────────────────────────────────────────────────────────
assert_eq(
    T, "volume_exato", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 4748,
    "dim_data deve ter exatamente 4748 linhas (2018-01-01 a 2030-12-31)")

# ─ Unicidade ──────────────────────────────────────────────────────────────
assert_zero(
    T, "pk_data_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT data) FROM {T}",
    "coluna 'data' (PK natural) contém duplicatas")
assert_zero(
    T, "pk_data_id_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT data_id) FROM {T}",
    "coluna 'data_id' (PK inteira) contém duplicatas")

# ─ Range e continuidade ────────────────────────────────────────────────
assert_zero(
    T, "data_anterior_2018", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE data < DATE '2018-01-01'",
    "existem datas anteriores a 2018-01-01")
assert_zero(
    T, "data_posterior_2030", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE data > DATE '2030-12-31'",
    "existem datas posteriores a 2030-12-31")
assert_zero(
    T, "sem_gaps_sequencia", "COMPLETENESS", "FAIL",
    f"""SELECT CASE
           WHEN COUNT(*) = DATEDIFF(DATE '2030-12-31', DATE '2018-01-01') + 1
           THEN 0 ELSE 1 END
       FROM {T}""",
    "sequência de datas possui lacunas")

# ─ Regras de negócio ──────────────────────────────────────────────────
assert_zero(
    T, "data_id_formato_yyyymmdd", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE data_id != CAST(date_format(data,'yyyyMMdd') AS INT)",
    "data_id não corresponde ao formato YYYYMMDD da coluna 'data'")
assert_zero(
    T, "ano_mes_formato_yyyy_mm", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE ano_mes != date_format(data,'yyyy-MM')",
    "ano_mes não corresponde ao formato YYYY-MM")
assert_zero(
    T, "is_fim_semana_correto", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE is_fim_semana != (DAYOFWEEK(data) IN (1,7))",
    "is_fim_semana inconsistente com DAYOFWEEK(data)")
assert_zero(
    T, "ultimo_dia_mes_correto", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE ultimo_dia_mes != LAST_DAY(data)",
    "ultimo_dia_mes inconsistente com LAST_DAY(data)")


───────────────────────────────────────────────────────────────────────────
  dim_data
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  volume_exato
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  pk_data_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  pk_data_id_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  data_anterior_2018
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  data_posterior_2030
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  sem_gaps_sequencia
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  data_id_formato_yyyymmdd
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  ano_mes_formato_yyyy_mm
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  is_fim_semana_correto
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_data  ultimo_dia_mes_correto


In [0]:
# ───────────────────────────────────────────────────────────────────────────
section("dim_operadora")
T = tbl("dim_operadora")
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 9999, "volume fora do intervalo esperado (deve ser > 0)")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT operadora_id) FROM {T}",
    "operadora_id contém duplicatas")
assert_zero(T, "operadora_nome_not_null", "COMPLETENESS", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE operadora_nome IS NULL",
    "operadora_nome contém valores NULL")

# ───────────────────────────────────────────────────────────────────────────
section("dim_empresa")
T = tbl("dim_empresa")
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 99999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT empresa_id) FROM {T}",
    "empresa_id contém duplicatas")
assert_zero(T, "empresa_nome_not_null", "COMPLETENESS", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE empresa_nome IS NULL",
    "empresa_nome contém valores NULL")
assert_zero(T, "porte_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE porte NOT IN ('Pequena','Média','Grande') AND porte IS NOT NULL",
    "porte fora do domínio: Pequena | Média | Grande")
assert_zero(T, "uf_len_2", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE uf IS NOT NULL AND LENGTH(uf) != 2",
    "uf com sigla de comprimento diferente de 2")
assert_zero(T, "data_inicio_not_future", "DOMAIN", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE data_inicio_relacionamento > CURRENT_DATE()",
    "data_inicio_relacionamento com datas no futuro")

# ───────────────────────────────────────────────────────────────────────────
section("dim_plano")
T = tbl("dim_plano")
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 99999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT plano_id) FROM {T}",
    "plano_id contém duplicatas")
assert_zero(T, "fk_operadora_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {tbl('dim_plano')} p
        WHERE p.operadora_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_operadora')} o WHERE o.operadora_id = p.operadora_id)""",
    "operadora_id com valores órfãos (sem correspondência em dim_operadora)")
assert_zero(T, "acomodacao_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE acomodacao NOT IN ('Enfermaria','Apartamento') AND acomodacao IS NOT NULL",
    "acomodacao fora do domínio: Enfermaria | Apartamento")
assert_zero(T, "coparticipacao_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE coparticipacao NOT IN ('Sim','Não') AND coparticipacao IS NOT NULL",
    "coparticipacao fora do domínio: Sim | Não")
assert_zero(T, "preco_vida_mes_positivo", "DOMAIN", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE preco_vida_mes IS NOT NULL AND preco_vida_mes <= 0",
    "preco_vida_mes com valor não positivo")

# ───────────────────────────────────────────────────────────────────────────
section("dim_corretor")
T = tbl("dim_corretor")
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 99999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT corretor_id) FROM {T}",
    "corretor_id contém duplicatas")
assert_zero(T, "senioridade_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE senioridade NOT IN ('Júnior','Pleno','Sênior') AND senioridade IS NOT NULL",
    "senioridade fora do domínio: Júnior | Pleno | Sênior")
assert_zero(T, "regiao_not_null", "COMPLETENESS", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE regiao IS NULL",
    "regiao contém valores NULL")
assert_zero(T, "data_admissao_not_future", "DOMAIN", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE data_admissao > CURRENT_DATE()",
    "data_admissao com datas no futuro")


───────────────────────────────────────────────────────────────────────────
  dim_operadora
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_operadora volume
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_operadora pk_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_operadora operadora_nome_not_null

───────────────────────────────────────────────────────────────────────────
  dim_empresa
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa volume
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa pk_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa empresa_nome_not_null
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa porte_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa uf_len_2
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_empresa data_inicio_not_future

─────────────────────────────────────

In [0]:
section("dim_beneficiario")
T = tbl("dim_beneficiario")

# ─ Volume e PK ───────────────────────────────────────────────────────────────
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 9_999_999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT beneficiario_id) FROM {T}",
    "beneficiario_id contém duplicatas")

# ─ Integridade referencial ───────────────────────────────────────────────
assert_zero(T, "fk_contrato_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} b
        WHERE b.contrato_id IS NOT NULL
        AND NOT EXISTS (
            SELECT 1 FROM {tbl('fat_contratos')} c WHERE c.contrato_id = b.contrato_id)""",
    "contrato_id com valores órfãos (sem correspondência em fat_contratos)")
assert_zero(T, "fk_empresa_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} b
        WHERE b.empresa_id IS NOT NULL
        AND NOT EXISTS (
            SELECT 1 FROM {tbl('dim_empresa')} e WHERE e.empresa_id = b.empresa_id)""",
    "empresa_id com valores órfãos (sem correspondência em dim_empresa)")

# ─ Domínio ────────────────────────────────────────────────────────────────
assert_zero(T, "tipo_beneficiario_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE tipo_beneficiario NOT IN ('Titular','Dependente') AND tipo_beneficiario IS NOT NULL",
    "tipo_beneficiario fora do domínio: Titular | Dependente")
assert_zero(T, "sexo_domain", "DOMAIN", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE sexo NOT IN ('Masculino','Feminino') AND sexo IS NOT NULL",
    "sexo fora do domínio: Masculino | Feminino (NULLs são esperados em ~19% dos registros)")
assert_zero(T, "situacao_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE situacao NOT IN ('Ativo','Cancelado')",
    "situacao fora do domínio: Ativo | Cancelado")
assert_zero(T, "faixa_etaria_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE faixa_etaria NOT IN ('00-17','18-29','30-44','45-59','60+') AND faixa_etaria IS NOT NULL",
    "faixa_etaria fora do domínio: 00-17 | 18-29 | 30-44 | 45-59 | 60+")

# ─ Regras de negócio ──────────────────────────────────────────────────
assert_zero(T, "situacao_consistente_com_data_cancelamento", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE (data_cancelamento IS NOT NULL AND situacao != 'Cancelado')
           OR (data_cancelamento IS     NULL AND situacao != 'Ativo')""",
    "situacao diverge da lógica: NULL → Ativo, preenchido → Cancelado")
assert_zero(T, "data_nascimento_nao_futura", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE data_nascimento > CURRENT_DATE()",
    "data_nascimento com datas no futuro")
assert_zero(T, "data_adesao_apos_nascimento", "BUSINESS_RULE", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE data_adesao < data_nascimento",
    "data_adesao anterior a data_nascimento")


───────────────────────────────────────────────────────────────────────────
  dim_beneficiario
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario volume
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario pk_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario fk_contrato_id
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario fk_empresa_id
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario tipo_beneficiario_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario sexo_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario situacao_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario faixa_etaria_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario situacao_consistente_com_data_cancelamento
  ✅ [PASS]  homologacao.salutar_saude.gold_dim_beneficiario data_nascimento_nao_futura
  ✅ [PASS]  homologacao.salutar_saude.g

In [0]:
section("fat_contratos")
T = tbl("fat_contratos")

# ─ Volume e PK ───────────────────────────────────────────────────────────────
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 9_999_999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT contrato_id) FROM {T}",
    "contrato_id contém duplicatas")

# ─ Integridade referencial ──────────────────────────────────────────────
assert_zero(T, "fk_empresa_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} c WHERE c.empresa_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_empresa')} e WHERE e.empresa_id = c.empresa_id)""",
    "empresa_id com valores órfãos em dim_empresa")
assert_zero(T, "fk_plano_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} c WHERE c.plano_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_plano')} p WHERE p.plano_id = c.plano_id)""",
    "plano_id com valores órfãos em dim_plano")
assert_zero(T, "fk_corretor_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} c WHERE c.corretor_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_corretor')} k WHERE k.corretor_id = c.corretor_id)""",
    "corretor_id com valores órfãos em dim_corretor")

# ─ Domínio ────────────────────────────────────────────────────────────────
assert_zero(T, "status_domain", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE status NOT IN ('Ativo','Cancelado','Renovado') AND status IS NOT NULL",
    "status fora do domínio: Ativo | Cancelado | Renovado")
assert_zero(T, "num_vidas_positivo", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE num_vidas IS NOT NULL AND num_vidas <= 0",
    "num_vidas com valor não positivo")
assert_zero(T, "valor_mensal_positivo", "DOMAIN", "WARN",
    f"SELECT COUNT(*) FROM {T} WHERE valor_mensal IS NOT NULL AND valor_mensal <= 0",
    "valor_mensal com valor não positivo (1 NULL na fonte é esperado)")

# ─ Regras de negócio ──────────────────────────────────────────────────
assert_zero(T, "vigencia_ordem_correta", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE vigencia_fim < vigencia_inicio",
    "vigencia_fim anterior a vigencia_inicio")
assert_zero(T, "receita_total_mensal_coerente", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE ABS(CAST(receita_total_mensal AS DOUBLE)
                  - COALESCE(CAST(valor_mensal AS DOUBLE) * num_vidas, 0)) > 0.01""",
    "receita_total_mensal diverge de COALESCE(valor_mensal * num_vidas, 0)")
assert_zero(T, "dias_vigencia_coerente", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE dias_vigencia != DATEDIFF(vigencia_fim, vigencia_inicio)""",
    "dias_vigencia diverge de DATEDIFF(vigencia_fim, vigencia_inicio)")


───────────────────────────────────────────────────────────────────────────
  fat_contratos
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos volume
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos pk_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos fk_empresa_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos fk_plano_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos fk_corretor_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos status_domain
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos num_vidas_positivo
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos valor_mensal_positivo
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos vigencia_ordem_correta
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos receita_total_mensal_coerente
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_contratos dias_vigencia_coerente


In [0]:
section("fat_utilizacao")
T = tbl("fat_utilizacao")

# ─ Volume e PK ───────────────────────────────────────────────────────────────
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 99_999_999, "volume fora do intervalo esperado")
assert_zero(T, "pk_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT evento_id) FROM {T}",
    "evento_id contém duplicatas")

# ─ Integridade referencial ──────────────────────────────────────────────
assert_zero(T, "fk_beneficiario_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} u WHERE u.beneficiario_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_beneficiario')} b WHERE b.beneficiario_id = u.beneficiario_id)""",
    "beneficiario_id com valores órfãos em dim_beneficiario")
assert_zero(T, "fk_contrato_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} u WHERE u.contrato_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('fat_contratos')} c WHERE c.contrato_id = u.contrato_id)""",
    "contrato_id com valores órfãos em fat_contratos")
assert_zero(T, "fk_empresa_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} u WHERE u.empresa_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_empresa')} e WHERE e.empresa_id = u.empresa_id)""",
    "empresa_id com valores órfãos em dim_empresa")
assert_zero(T, "fk_plano_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} u WHERE u.plano_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_plano')} p WHERE p.plano_id = u.plano_id)""",
    "plano_id com valores órfãos em dim_plano")

# ─ Completude e domínio ───────────────────────────────────────────────
assert_zero(T, "data_evento_not_null", "COMPLETENESS", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE data_evento IS NULL",
    "data_evento contém valores NULL")
assert_zero(T, "valor_sinistro_not_null", "COMPLETENESS", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE valor_sinistro IS NULL",
    "valor_sinistro contém valores NULL")
assert_zero(T, "valor_sinistro_positivo", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE valor_sinistro <= 0",
    "valor_sinistro com valor não positivo")
assert_zero(T, "tipo_evento_known", "DOMAIN", "WARN",
    f"""SELECT COUNT(*) FROM {T}
        WHERE tipo_evento NOT IN ('Consulta','Exame','Cirurgia','Pronto-socorro','Internação')
        AND tipo_evento IS NOT NULL""",
    "tipo_evento fora da lista conhecida (lista aberta — pode ser esperado)")
assert_zero(T, "data_evento_in_dim_data_range", "INTEGRITY", "WARN",
    f"""SELECT COUNT(*) FROM {T}
        WHERE data_evento < DATE '2018-01-01' OR data_evento > DATE '2030-12-31'""",
    "data_evento fora do range coberto por dim_data (2018–2030)")


───────────────────────────────────────────────────────────────────────────
  fat_utilizacao
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao volume
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao pk_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao fk_beneficiario_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao fk_contrato_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao fk_empresa_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao fk_plano_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao data_evento_not_null
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao valor_sinistro_not_null
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_utilizacao valor_sinistro_positivo
  ⚠️  [WARN]  homologacao.salutar_saude.gold_fat_utilizacao tipo_evento_known
           ↳ tipo_evento fora da lista conhecida (lista aberta — pode ser esperado) 

In [0]:
section("fat_sinistralidade")
T = tbl("fat_sinistralidade")

# ─ Volume e PK composta ────────────────────────────────────────────────
assert_between(T, "volume", "VOLUME", "FAIL",
    f"SELECT COUNT(*) FROM {T}", 1, 999_999, "volume fora do intervalo esperado")
assert_zero(T, "pk_composta_unique", "UNIQUENESS", "FAIL",
    f"SELECT COUNT(*) - COUNT(DISTINCT CONCAT(CAST(empresa_id AS STRING),'|',ano_mes)) FROM {T}",
    "chave composta (empresa_id, ano_mes) contém duplicatas")

# ─ Integridade referencial ──────────────────────────────────────────────
assert_zero(T, "fk_empresa_id", "INTEGRITY", "FAIL",
    f"""SELECT COUNT(*) FROM {T} s WHERE s.empresa_id IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {tbl('dim_empresa')} e WHERE e.empresa_id = s.empresa_id)""",
    "empresa_id com valores órfãos em dim_empresa")
assert_zero(T, "fk_ano_mes_em_dim_data", "INTEGRITY", "WARN",
    f"""SELECT COUNT(*) FROM {T} s
        WHERE NOT EXISTS (SELECT 1 FROM {tbl('dim_data')} d WHERE d.ano_mes = s.ano_mes)""",
    "ano_mes com valores sem correspondência em dim_data.ano_mes")

# ─ Domínio de medidas ────────────────────────────────────────────────
assert_zero(T, "receita_nao_negativa", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE receita_premios < 0",
    "receita_premios com valores negativos")
assert_zero(T, "custo_nao_negativo", "DOMAIN", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE custo_sinistros < 0",
    "custo_sinistros com valores negativos")

# ─ Regra P1/P2: sinistralidade nula somente quando receita = 0 ─────────────
assert_zero(T, "P1_sinistralidade_not_null_when_receita_gt_zero", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE receita_premios > 0 AND sinistralidade IS NULL",
    "[P1] sinistralidade é NULL com receita_premios > 0")
assert_zero(T, "P4_sinistralidade_null_when_receita_zero", "BUSINESS_RULE", "FAIL",
    f"SELECT COUNT(*) FROM {T} WHERE receita_premios = 0 AND sinistralidade IS NOT NULL",
    "[P4] sinistralidade não-NULL com receita_premios = 0")

# ─ Regra: coerência das colunas derivadas ──────────────────────────────
assert_zero(T, "margem_tecnica_coerente", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE ABS(CAST(margem_tecnica AS DOUBLE)
               - (CAST(receita_premios AS DOUBLE) - CAST(custo_sinistros AS DOUBLE))) > 0.01""",
    "margem_tecnica diverge de receita_premios - custo_sinistros")
assert_zero(T, "sinistralidade_pct_coerente", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE sinistralidade IS NOT NULL
        AND ABS(CAST(sinistralidade_pct AS DOUBLE) -
                ROUND(CAST(sinistralidade AS DOUBLE) * 100, 2)) > 0.01""",
    "sinistralidade_pct diverge de sinistralidade * 100")
assert_zero(T, "P4_flag_sinistro_sem_receita_coerente", "BUSINESS_RULE", "FAIL",
    f"""SELECT COUNT(*) FROM {T}
        WHERE sinistro_sem_receita != (receita_premios = 0 AND custo_sinistros > 0)""",
    "[P4] flag sinistro_sem_receita inconsistente com receita_premios e custo_sinistros")

# ─ Informação sobre registros P4 ───────────────────────────────────────
print()
n_p4 = spark.sql(f"SELECT COUNT(*) FROM {T} WHERE sinistro_sem_receita = TRUE").collect()[0][0]
if n_p4 > 0:
    print(f"  ⚠️   [INFO]  {T}")
    print(f"           {n_p4} registro(s) com sinistro_sem_receita=TRUE [P4] — investigue as empresas abaixo:")
    spark.sql(f"""
        SELECT empresa_id, empresa_nome, ano_mes, custo_sinistros
        FROM {T}
        WHERE sinistro_sem_receita = TRUE
        ORDER BY custo_sinistros DESC
        LIMIT 10
    """).show(truncate=False)
else:
    print(f"  ✅  [INFO]  {T}: nenhum registro com sinistro_sem_receita=TRUE")


───────────────────────────────────────────────────────────────────────────
  fat_sinistralidade
───────────────────────────────────────────────────────────────────────────
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade volume
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade pk_composta_unique
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade fk_empresa_id
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade fk_ano_mes_em_dim_data
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade receita_nao_negativa
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade custo_nao_negativo
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade P1_sinistralidade_not_null_when_receita_gt_zero
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade P4_sinistralidade_null_when_receita_zero
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinistralidade margem_tecnica_coerente
  ✅ [PASS]  homologacao.salutar_saude.gold_fat_sinis

In [0]:
section("SUMÁRIO FINAL")

n_pass  = sum(1 for r in _RESULTS if r.status == "PASS")
n_warn  = sum(1 for r in _RESULTS if r.status == "WARN")
n_fail  = sum(1 for r in _RESULTS if r.status == "FAIL")
n_total = len(_RESULTS)

print(f"\n  Total de testes  : {n_total}")
print(f"  ✅  PASS          : {n_pass}")
print(f"  ⚠️   WARN          : {n_warn}")
print(f"  ❌  FAIL          : {n_fail}")

if n_fail > 0:
    print("\n  Testes com FAIL:")
    for r in _RESULTS:
        if r.status == "FAIL":
            print(f"    ❌ [{r.category:<15}] {r.table} — {r.test}")
            print(f"        ↳ {r.detail}")

if n_warn > 0:
    print("\n  Testes com WARN:")
    for r in _RESULTS:
        if r.status == "WARN":
            print(f"    ⚠️  [{r.category:<15}] {r.table} — {r.test}")
            print(f"        ↳ {r.detail}")

print(f"\n  Finalizado : {datetime.now():%Y-%m-%d %H:%M:%S}")

# ─ DataFrame interativo com todos os resultados ────────────────────────────────
from pyspark.sql import Row
from pyspark.sql.functions import when, col as _col

df_results = spark.createDataFrame([
    Row(status=r.status, category=r.category, severity=r.severity,
        table=r.table, test=r.test, detail=r.detail)
    for r in _RESULTS
])

# FAIL primeiro, WARN no meio, PASS por último
df_results = (
    df_results
    .withColumn("_ord",
        when(_col("status") == "FAIL", 0)
        .when(_col("status") == "WARN", 1)
        .otherwise(2))
    .orderBy("_ord", "table", "test")
    .drop("_ord")
)

display(df_results)

# ─ Bloqueia em caso de falhas críticas ─────────────────────────────────────────
if n_fail > 0:
    raise AssertionError(
        f"DQ Suite FALHOU: {n_fail} teste(s) crítico(s). "
        "Corrija as tabelas gold antes de prosseguir."
    )
else:
    print(f"\n  ✅ DQ Suite concluída com sucesso — todas as tabelas gold aprovadas.")


───────────────────────────────────────────────────────────────────────────
  SUMÁRIO FINAL
───────────────────────────────────────────────────────────────────────────

  Total de testes  : 74
  ✅  PASS          : 73
  ⚠️   WARN          : 1
  ❌  FAIL          : 0

  Testes com WARN:
    ⚠️  [DOMAIN         ] homologacao.salutar_saude.gold_fat_utilizacao — tipo_evento_known
        ↳ tipo_evento fora da lista conhecida (lista aberta — pode ser esperado)  [n=1,838]

  Finalizado : 2026-07-04 21:13:20


status,category,severity,table,test,detail
WARN,DOMAIN,WARN,homologacao.salutar_saude.gold_fat_utilizacao,tipo_evento_known,"tipo_evento fora da lista conhecida (lista aberta — pode ser esperado) [n=1,838]"
PASS,BUSINESS_RULE,WARN,homologacao.salutar_saude.gold_dim_beneficiario,data_adesao_apos_nascimento,data_adesao anterior a data_nascimento [n=0]
PASS,DOMAIN,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,data_nascimento_nao_futura,data_nascimento com datas no futuro [n=0]
PASS,DOMAIN,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,faixa_etaria_domain,faixa_etaria fora do domínio: 00-17 | 18-29 | 30-44 | 45-59 | 60+ [n=0]
PASS,INTEGRITY,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,fk_contrato_id,contrato_id com valores órfãos (sem correspondência em fat_contratos) [n=0]
PASS,INTEGRITY,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,fk_empresa_id,empresa_id com valores órfãos (sem correspondência em dim_empresa) [n=0]
PASS,UNIQUENESS,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,pk_unique,beneficiario_id contém duplicatas [n=0]
PASS,DOMAIN,WARN,homologacao.salutar_saude.gold_dim_beneficiario,sexo_domain,sexo fora do domínio: Masculino | Feminino (NULLs são esperados em ~19% dos registros) [n=0]
PASS,BUSINESS_RULE,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,situacao_consistente_com_data_cancelamento,"situacao diverge da lógica: NULL → Ativo, preenchido → Cancelado [n=0]"
PASS,DOMAIN,FAIL,homologacao.salutar_saude.gold_dim_beneficiario,situacao_domain,situacao fora do domínio: Ativo | Cancelado [n=0]



  ✅ DQ Suite concluída com sucesso — todas as tabelas gold aprovadas.
